# 01 – Load the Olist CSVs into MySQL

Creates the `olist` database, loads the 8 CSV files as tables (dates converted to `DATETIME`,
ID columns stored as `VARCHAR` so they can be indexed), then runs the data-quality checks and creates
the indexes and the clean `sales` view.

Credentials are read from `../.env` (see `.env.example`).

In [1]:
import pandas as pd
from sqlalchemy import text, types
from db import get_engine, run_sql_file

# 1. Create the database
with get_engine(database=None).begin() as conn:
    conn.execute(text("CREATE DATABASE IF NOT EXISTS olist"))
engine = get_engine("olist")

In [2]:
files = {
    "orders": "olist_orders_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

load_report = []
for table, file in files.items():
    df = pd.read_csv(f"../data/{file}", encoding="utf-8-sig")
    # turn every column that looks like a date into a real datetime
    for col in df.columns:
        if "timestamp" in col or "date" in col or col.endswith("_at"):
            df[col] = pd.to_datetime(df[col], errors="coerce")
    # short text columns -> VARCHAR (indexable), long text (review comments) -> TEXT
    dtype = {}
    for col in [c for c in df.columns if pd.api.types.is_string_dtype(df[c])]:
        max_len = int(df[col].astype(str).str.len().max())
        dtype[col] = types.VARCHAR(max(max_len, 1) + 10) if max_len <= 250 else types.TEXT()
    df.to_sql(table, engine, if_exists="replace", index=False, chunksize=5000, dtype=dtype)
    load_report.append({"table": table, "rows": len(df), "missing_values": int(df.isna().sum().sum())})
    print(f"{table}: {len(df):,} rows, {df.isna().sum().sum():,} missing values")

pd.DataFrame(load_report)

orders: 99,441 rows, 4,908 missing values


customers: 99,441 rows, 0 missing values


order_items: 112,650 rows, 0 missing values


order_payments: 103,886 rows, 0 missing values


order_reviews: 99,224 rows, 145,903 missing values


products: 32,951 rows, 2,448 missing values
sellers: 3,095 rows, 0 missing values


category_translation: 71 rows, 0 missing values


,table,rows,missing_values
0,orders,99441,4908
1,customers,99441,0
2,order_items,112650,0
3,order_payments,103886,0
4,order_reviews,99224,145903
5,products,32951,2448
6,sellers,3095,0
7,category_translation,71,0


## Data-quality checks (`sql/01_data_checks.sql`)

In [3]:
from db import split_sql
for stmt in split_sql("../sql/01_data_checks.sql")[1:]:   # skip "USE olist"
    print(stmt.splitlines()[0][:80], "...")
    display(pd.read_sql(text(stmt), engine))

SELECT order_status, COUNT(*) AS orders ...


,order_status,orders
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


SELECT order_id, COUNT(*) AS c ...


,order_id,c


SELECT COUNT(*) AS delivered_without_date ...


,delivered_without_date
0,8


... (4 outputs left out)

SELECT COUNT(DISTINCT p.product_category_name) AS untranslated_categories ...


,untranslated_categories
0,2


SELECT MIN(order_purchase_timestamp) AS first_order, ...


,first_order,last_order
0,2016-09-04 21:15:19,2018-10-17 17:30:18


## Indexes + clean `sales` view (`sql/02_indexes_and_view.sql`)

In [4]:
run_sql_file(engine, "../sql/02_indexes_and_view.sql")
print(pd.read_sql(text("SELECT COUNT(*) AS rows_in_sales_view FROM sales"), engine))
pd.read_sql(text("SELECT * FROM sales LIMIT 5"), engine)

   rows_in_sales_view
0              110197


,order_id,customer_unique_id,customer_state,purchase_time,month,product_id,seller_id,price,freight_value,category,delivery_days,is_late
0,e481f51cbdc54678b7cc49136f2d6af7,7c396fd4830fd04220f754e42b4e5bff,SP,2017-10-02 10:56:33,2017-10,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,29.99,8.72,housewares,8,0
1,53cdb2fc8bc7dce0b6741e2150273451,af07308b275d755c9edb36a90c618231,BA,2018-07-24 20:41:37,2018-07,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,118.70,22.76,perfumery,14,0
2,47770eb9100c2d0c44946d9cf07ec65d,3a653a41f6f9fc3d2a113cf8398680e8,GO,2018-08-08 08:38:49,2018-08,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,159.90,19.22,auto,9,0
3,949d5b44dbf5de918fe9c16f97b45f8a,7c142cf63193a1473d2e66489a9ae977,RN,2017-11-18 19:28:06,2017-11,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,45.00,27.20,pet_shop,14,0
4,ad21c59c0840e6cb83a9ceb5573f8159,72632f0f9dd73dfee390c9b22eb56dd6,SP,2018-02-13 21:18:39,2018-02,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,19.90,8.72,stationery,3,0
